---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-39: Agentic AI and Tool Calling - I</h1>

# Learning agenda of this notebook
1. Limitations of LLMs 
2. From Simple LLM apps to Agents and Agentic AI Systems
    - Overview
    - Key Components of an Agentic AI System
3. Workflow of a Tool Call
    - Workflow of a Tool Call
    - Flow Description: Web Search Tool Example
4. Calling OpenAI's Hosted **Built-in Tools** using the OpenAI's `Responses` API
   - Example: Using tool `web_search`
   - Example: Using tool `code_interpreter` with auto container
   - Example: Using tool `code_interpreter` with explicit container
   - Example: Using tool `file_search`
5. Calling Groq's Hosted **Built-in Tools** using the OpenAI's `Chat Completions` API
    - Example: Using Groq's Built-in tool `browser_search`
    - Example: Using Groq's Built-in tool `browser_search` to read a specific web page
    - Example: Using Groq's Built-in tool `code_interpreter`
6. Calling OpenRouter's Hosted **Server Tools** using the OpenAI's `Chat Completions` API
    - Example: Using OpenRouter's server tool `openrouter:web_search`
    - Example: Using OpenRouter's server tool `openrouter:web_fetch` to read a specific web page
    - Example: Combining `openrouter:datetime` and `openrouter:web_search` with a non-OpenAI model

# <span style='background :lightgreen' >1. Limitations of LLMs</span>
## a. Knowledge Cutoff:
- LLMs operate within fixed training data boundaries, with a specific knowledge cutoff date. This temporal limitation prevents models from accessing information about events, developments, or data that occurred after their training completion.
-   Without tools LLMs CANNOT
    - ❌ Tell you the current date and time (e.g., "What is today's date?")
    - ❌ Access recent news or events (e.g., "Who won yesterday's cricket match between Pakistan and India?")
    - ❌ Provide information about products/services launched after training (e.g., "What are the specs of the iPhone that Apple launched last month?")
    - ❌ Know about software updates, new research papers, or policy changes (e.g., "What's the latest version of Python released this year?")
    - ❌ Answer questions about current trends (e.g., "What's trending on X (formerly Twitter) right now?")
- Function calling enables models to search the web for current information, query external knowledge sources and databases, and access live APIs for up-to-date data
## b. No Real-Time Awareness:
- Pre-trained language models cannot access dynamic, real-time information such as current conditions, live data streams, or system states that change moment-to-moment.
- Without tools LLMs CANNOT
    - ❌ Check current weather conditions (e.g., "What's the weather in Lahore right now?")
    - ❌ Get live stock prices or cryptocurrency values (e.g., "What's the current price of Bitcoin?")
    - ❌ Access breaking news or real-time events (e.g., "Is there any breaking news about the earthquake in Afghanistan?")
    - ❌ Check flight schedules, delays, or availability (e.g., "Is flight PK-301 from Karachi to Dubai on time?")
    - ❌ Track package deliveries or order status (e.g., "Where is my Amazon package with tracking number XYZ123?")
    - ❌ Get sports scores or live game updates (e.g., "What's the current score in the PSL final?")
    - ❌ Check current traffic conditions (e.g., "Is there traffic on the route from Gulberg to DHA Lahore?")
- Function calling enables models to retrieve live data from APIs (weather, stocks, flights), make reservations and bookings, access real-time external services, query sensors and IoT devices and so on.
## c. No Access to Private Data:
- LLMs cannot access private, proprietary, or authenticated systems such as company databases, internal documents, personal accounts, or enterprise applications. This is both a security feature and a limitation.
- Without tools LLMs CANNOT:
    - ❌ Access your Google Drive documents or Gmail (e.g., "Find the Q3 sales report in my Google Drive and summarize it")
    - ❌ Query your GitHub repositories or pull requests (e.g., "What open pull requests do I have in my 'ecommerce-app' repository?")
    - ❌ Read Slack messages or team conversations (e.g., "What did the engineering team discuss in #backend channel today?")
    - ❌ Access databases (SQLite, PostgreSQL, MySQL, MongoDB) (e.g., "Show me all customers who made purchases over \$1000 last month from our MySQL database" )
    - ❌ Read your Notion pages or Confluence wiki (e.g., "What's written in my 'Project Timeline' page on Notion?")
    - ❌ Access project management tools (Jira, Asana, Trello)  (e.g., "Create a new Jira ticket: 'Fix login bug' assigned to Ahmad with priority High")
- Function calling provides a secure, authenticated way to access sensitive data using OAuth authentication for cloud services (Google, Microsoft), API keys and tokens for secure connections, and encrypted data transmission.
## d. No Execution Capability:
- Complex computational tasks that require specialized algorithms, intensive mathematical computations, or processing beyond natural language understanding necessitate external tool integration.
- Without tools LLMs CANNOT:
    - ❌ Perform complex mathematical or scientific computations accurately.   (e.g., "Calculate the standard deviation of this dataset: `[12, 15, 18, 22, 25, 28, 30]` and create a histogram")
    - ❌ Access the local file system of your machine. (e.g., "Create a file named abc.txt on the desktop of my machine and write a string "Hello Tools" inside it")
    - ❌ Compile/execute programs or run commands on your operating system. (e.g., "Run this Python script and show me the output")
    - ❌ Process images, videos, or audio files (e.g., "Resize this image to 800x600 pixels and apply a grayscale filter")
    - ❌ Perform database transactions (e.g., "Insert a new record into the customers table with name='Ali Khan' and email='ali@example.com'")
    - ❌ Generate visualizations or charts (e.g., "Create a pie chart showing the distribution of sales across regions")
- Function calling enables models to execute code in sandboxed environments, perform mathematical computations using libraries like NumPy and SciPy, access file systems, process media files, and generate visualizations.

# <span style='background :lightgreen' >2. From Simple LLM apps to Agents  and Agentic AI Systems</span>
# a. Overview

<div style="text-align:center;">
    <img src="../images/agenticai1.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>





<h2 align="center"><div class="alert alert-success" style="margin: 20px">A <strong>Simple LLM Application</strong> makes a single LLM call per request. The developer fixes the flow in advance: the model takes the prompt and any supplied context (system instructions, conversation history, user-provided documents) and generates a text response in one pass. It does not call tools, take actions, or loop. Any reasoning happens inside that single generation.</div></h2>

<div style="margin: 20px 40px; color: #7e22ce; text-align: left;">
<strong>Common use cases:</strong>
<ol>
<li><strong>Grammar and tone checker:</strong> Paste a draft email or essay and get back a corrected, more professional version (the Grammarly-style rewrite).</li>
<li><strong>Plain chatbot, summarizer or translator:</strong> Ask a general question, summarize a long article, or translate a paragraph into Urdu, all answered in one call.</li>
</ol>
</div>


<h2 align="center"><div class="alert alert-success" style="margin: 20px">An <strong>LLM Workflow</strong> is a system in which developer-written code orchestrates one or more LLM calls, and often tools or retrieval steps, along a predefined path. Common patterns include prompt chaining, routing, parallelization, and evaluator-optimizer loops. The steps and branching logic are decided at design time, so the LLM fills in each step but does not decide what the next step is. RAG pipelines and extract → validate → summarize pipelines are typical examples.</div></h2>

<div style="margin: 20px 40px; color: #7e22ce; text-align: left;">
<strong>Common use cases:</strong>
<ol>
<li><strong>Customer-support bot over company docs (RAG):</strong> Every question follows the same path: search the knowledge base → pass the top results to the LLM → return an answer with sources.</li>
<li><strong>Support-ticket router:</strong> One LLM call classifies each ticket as billing, technical, or general. The code then sends it to the matching prompt template, which drafts a reply. The LLM never chooses the route; the developer's if/else logic does.</li>
</ol>
</div>


<h2 align="center"><div class="alert alert-success" style="margin: 20px">An <strong>Agent</strong> is an LLM-based system in which the model itself controls the flow of execution. In a loop, it reasons about a goal, chooses and calls tools, observes the results, and adapts its next steps until it reaches a stopping condition: the goal is met, a step or cost budget is exhausted, or control is handed to a human. Patterns such as ReAct (Reason + Act) implement this loop. The degree of autonomy varies with the guardrails, permissions, and human approval gates the developer puts in place.</div></h2>

<div style="margin: 20px 40px; color: #7e22ce; text-align: left;">
<strong>Common use cases:</strong>
<ol>
<li><strong>Coding assistant (Claude Code, GitHub Copilot agent mode):</strong> Given "fix this failing test," it reads files, edits code, runs the tests, studies the errors, and retries until they pass.</li>
<li><strong>Deep research or shopping agent:</strong> Given "find the best laptop under $1,000" or "compare the top vector databases," it searches, opens pages, filters results, and decides when it has enough information.</li>
</ol>
</div>


<h2 align="center"><div class="alert alert-success" style="margin: 20px">An <strong>Agentic AI System</strong> is a coordinated network of multiple specialized AI agents that work together through an orchestration layer, sharing memory and communicating with each other to autonomously accomplish complex, long-running objectives with minimal human intervention.</div></h2>

<div style="margin: 20px 40px; color: #7e22ce; text-align: left;">
<strong>Common use cases:</strong>
<ol>
<li><strong>Voice-based customer support team (e.g., bank, airline, or telecom helpline):</strong> A customer calls and says, "I was charged twice for my plan." A greeter agent turns the speech into text, works out the intent, and routes the call. A verification agent confirms the caller's identity, and a billing agent checks the account and issues the refund. A voice agent speaks each reply naturally. All of them share the caller's history and the conversation so far, so the customer never has to repeat themselves. If the case is complex or the caller is upset, an escalation agent hands the call to a human with a short summary.</li>
<li><strong>AI research team (e.g., Claude's Research feature):</strong> A user asks, "Compare the battery strategies of the top 3 EV makers." A lead agent plans the work and splits it into separate questions. Several search agents work in parallel, one per company or source type, and save their findings to a shared notebook. A fact-checker agent verifies the claims and sources, and the lead agent merges everything into one well-organized report. If a section is weak or has gaps, the lead agent sends the agents back out for more.</li>
</ol>
</div>

# b. Key Components of an Agentic AI System

<div style="text-align:center;">
    <img src="../images/agenticai2.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## (i) LLM(s) (The planning, reasoning & decision making engine)
#### LLMs **without Tool Calling Capability**
- Basic text generation only (One-shot: single prompt → response)
- Cannot interact with external APIs or functions
- Suitable for: Simple Q&A, content generation, summarization
- **Example Models:**
    - OpenAI's GPT-2, GPT-3 base models
    - Anthropic's Claude 1.x, Claude 2.x (basic versions)
    - Google's PaLM 2 (text-bison)
    - Meta's Llama 1, Llama 2 base models
    - DeepSeek LLM (base) and DeepSeek-Coder (basic)
    - Mistral 7B base
    - Alibaba's Qwen 1.0 base models
    - Microsoft Phi-1, Phi-2

#### LLMs **With Tool/Function Calling** Capability
- Can invoke external APIs and tools using a standardized JSON schema to define tool signatures
- The model autonomously decides when and which tools to call based on the user's prompt
- Returns structured output containing tool name, arguments, and optionally parallel tool calls
- Your application executes the tool and returns results back to the model to generate the final response
- Suitable for: data retrieval, calculations, web search, API integration, and agentic workflows
- **Example Models:**
    - **OpenAI:** GPT-4o, GPT-4o-mini, GPT-4.1 (mini/nano), o3, o4-mini, the GPT-5.x series (GPT-5 through GPT-5.5), GPT-5.6 (Luna/Sol/Terra), and the current GPT-6 family (GPT-6 Astra, GPT-6 Sol, GPT-6 Luna, GPT-6.1 Sol). All support tool calling; GPT-5.4+ additionally supports `tool_search` for deferred tool loading, and the GA `computer` tool replaced `computer-use-preview` (shut down July 2026)
    - **Anthropic (Claude):** Claude 3 (Opus/Sonnet/Haiku), Claude 3.5 (Sonnet/Haiku), Claude 3.7 Sonnet, Claude 4.x (Opus/Sonnet/Haiku 4 through Claude Opus 4.7), and the Claude 5 family: Claude Fable 5.1 *(current top tier)*, Claude Opus 5.5, Claude Sonnet 5.5 and Claude Haiku 5.5. All support both client tools (user-defined) and server tools (web search, code execution)
    - **Google (Gemini):** Gemini 1.0 Pro (first Gemini model with function calling, Dec 2023), Gemini 2.5 Pro/Flash/Flash-Lite, Gemini 3 Flash, Gemini 3.1 Pro, Gemini 3.1 Flash/Flash-Lite, and Gemini 3.8 Flash *(newest stable model)*
    - **Llama by Meta (USA):** Llama 3.1 (8B/70B/405B), Llama 3.2, Llama 3.3 70B, Llama 4 Scout (109B total / 17B active, 10M context), Llama 4 Maverick (400B total / 17B active, 1M context). Llama 4 (April 2025) is Meta's latest open-weight release; its newer frontier model, Muse Spark (April 2026), is proprietary
    - **DeepSeek:** DeepSeek-V3 (671B MoE), DeepSeek-V3.2 (685B MoE), DeepSeek-Coder-V2, DeepSeek-V4 Flash (284B / 13B active), DeepSeek-V4 Pro (1.6T / 49B active)
    - **Mistral:** Mixtral 8x7B, Mixtral 8x22B, Mistral NeMo (12B), Mistral Large 2, Mistral Small 3.1/3.2 (24B), Devstral 2 (123B), Mistral Large 3 (675B MoE), Mistral Small 4 (119B MoE, Apache 2.0, Mar 2026)
    - **Alibaba (Qwen):** Qwen 2.5 (0.5B–72B), Qwen 2.5-Coder, Qwen 3 (up to 235B MoE), Qwen 3-Coder, Qwen 3.5 (up to 397B MoE), Qwen 3.6 (open-weight 27B and 35B-A3B; Qwen3.6-Plus is API-only), Qwen 3.8 (open weights, Aug 2026). Strong agentic tool-calling across all sizes; most open-weight Qwen models are Apache 2.0, so check each model card for its license
    - **Microsoft (Phi):** Phi-3 (3.8B/7B/14B), Phi-3.5, Phi-4 (14B), Phi-4-mini (3.8B), Phi-4-reasoning (14B, MIT license), Phi-4-reasoning-vision (15B). These are compact models optimized for local deployment, with solid tool-calling support
    - **Kimi by Moonshot AI (China):** Kimi K2.5 and K2.6 are ~1 trillion parameter MoE models with 32B active parameters, 256K context, and strong agentic/tool-calling capabilities, available in Ollama via `ollama run kimi-k2.5` or `kimi-k2.5:cloud`. Its newest flagship, Kimi K3, was launched in July 2026, with open weights under a custom Kimi K3 license.
    - **MiniMax by MiniMax AI (China):** MiniMax M2.7 is a 230B MoE model with only 10B active parameters, released on Hugging Face in April 2026, with strong agentic tool-calling capabilities. It is supported locally in Ollama as `minimax-m2.1` and via cloud as `minimax-m2.7`.
    - **GLM by Z.ai (China):** GLM-4.7 and GLM-5.1 range from 355B to 744B MoE parameters (32–40B active). GLM-5.1 weights were published on April 7, 2026 under an MIT license, designed for long-horizon agentic tasks and sustained tool calling, followed by GLM-5.2 in June 2026. GLM is supported locally in Ollama as `glm-4.7` and via cloud as `glm-5.1:cloud`.

#### LLMs **With Reasoning Capability**
- Uses extended thinking or chain-of-thought before responding
- Spends compute on internal reasoning steps
- Better at: Math, coding, logic puzzles, scientific analysis
- Higher latency but better quality for complex tasks
- **Example Models:**
    - OpenAI's o1, o1-mini, o1-pro, o3, o3-mini, o3-pro, o4-mini; GPT-5 → GPT-5.5 (with configurable `reasoning.effort`), GPT-5.x Pro, and the GPT-6 family (GPT-6 Astra, GPT-6 Sol, GPT-6 Luna, GPT-6.1 Sol)
    - Anthropic's Claude 3.7 Sonnet and Claude 4.x (extended thinking), and the Claude 5 family: Claude Fable 5.1, Claude Opus 5.5, Claude Sonnet 5.5, Claude Haiku 5.5 (adaptive thinking with configurable effort)
    - Google's Gemini 2.5 Pro/Flash (thinking mode), Gemini 2.5 Deep Think, Gemini 3 Pro (Deep Think mode), Gemini 3.1 Pro, Gemini 3.8 Flash (configurable thinking level)
    - DeepSeek-R1 (671B, full reasoning), DeepSeek-R1-0528 (updated), DeepSeek-R1 Distilled models (Qwen 1.5B/7B/14B/32B, Llama 8B/70B), DeepSeek-V4 Pro/Flash (hybrid thinking/non-thinking modes)
    - Microsoft Phi-4-reasoning (14B), Phi-4-reasoning-plus (14B, SFT+RL), Phi-4-mini-reasoning (3.8B, math-focused), Phi-4-reasoning-vision (15B, multimodal reasoning, March 2026)
    - Mistral's Magistral Small/Medium (dedicated reasoning models), whose reasoning is now merged into Mistral Small 4 with configurable reasoning depth
    - Alibaba's Qwen 3 (235B-A22B, hybrid thinking/non-thinking) and later Qwen releases, where thinking is toggled with the `enable_thinking` flag and, per turn, with the `/think` and `/no_think` soft switches


## (ii) Memory (Remembers  the context, past history and user preferences)
- Memory enables an agentic AI system to retain context, learn from past interactions, and behave consistently over time. Memory transforms a simple agent into an intelligent agent or assistant that actually understands you and your needs. Without memory every conversation with your agent starts from zero. It is like talking to someone with amnesia. With memory your agent remembers previous conversations, user preferences, past decisions and historical context. Types of memory that an agent uses are:
- **Short-term memory** in LLM applications is the information it has about recent interactions, typically the ongoing conversations with the user or the behavior of the LLM. In LLM applications, this manifests as the conversation history that persists across API calls and is continuously provided to the model as context.
    - **Key Characteristics:**
        - *Session-Based Persistence:* Memory exists only during an active session and must be re-fed with each interaction
        - *Context Window Constraints:* Implemented as conversation history within the prompt, constrained by the model's token limit
        - *Dynamic Management:* Requires active strategies to manage limited space as conversations grow
    - **Basic Implementation Technique (Send Full Conversation History with each API Call):** Stores and transmits the complete history of user messages and assistant responses with each API call.
        - If using *`Chat Completion API`* you maintain a list of all messages and include them in every request (rapidly consumes token budget).
        - If using *`Responses API`* you can pass `previous_response_id` (chains to the previous response) or `conversation` (a durable conversation object created with the Conversations API), and OpenAI keeps the history on the server side. The earlier turns are still sent to the model, so they still count (and are billed) as input tokens.
    - **Common Techniques to Keep the  History within the Model's Context Window:**
        - *Sliding Window:* Retains only the most recent N messages, creating a "moving window" through the conversation. It can be implemented using a fixed-size buffer (e.g., FIFO queue) that automatically discards the oldest message when a new one arrives. It maintains recent context and is efficient for ongoing conversations. However, earlier details and context are permanently lost; may lose critical information from the beginning of conversations.
        - *Token-Based Truncation:* Manages memory based on token consumption rather than message count, maintaining a "token budget." It can be implemented by tracking cumulative token count of messages; when approaching the limit, remove oldest messages until within budget. This technique also discards important early information and requires token counting logic.
        - *Context Switching:* Dynamically adjusts focus based on detected topic changes, prioritizing relevant context. It can be implemented using topic modeling or classification to detect conversation shifts. The programmer needs to maintain separate context buffers for different topics or reset context when topics change. For example: When conversation shifts from travel planning to cooking recipes, the system deprioritizes flight details and focuses on cooking recipes.
        -  *Conversation Summarization:* Generates compressed summaries of older conversation segments, similar to meeting notes. It can be implemented by periodically invoking the LLM to create concise summaries of older exchanges; replace original messages with summaries to save tokens. It requires additional API calls to LLM for summary generation and the risk of losing granular details still exists.
- **Long-term memory** is critical for building autonomous and intelligent agentic AI systems. Unlike short-term memory (which exists only during a session), it has following characteristics:
    - Persists across multiple sessions, conversations, and interactions, enabling agents to recall past interactions
    - Should store information like user name, his age, likes and dislikes
    - Should store decisions made in the past
    - Should evolve over time implicitly or explicitly.
- **Three Types of Long-Term Memory (Parametric is not one of them)**
    - **Parametric Memory (Model Weights):** Stores world facts, skills, rules and reasoning patterns inside model weights during training (e.g., knowing Islamabad is the capital of Pakistan, or being able to write Python code). Useful for global knowledge, not for personalized or evolving state. Changing it requires touching the weights — continued pretraining, fine-tuning (full or LoRA/adapters), RLHF/DPO, or direct model editing (ROME/MEMIT) — so it cannot be updated at runtime, is opaque and un-auditable, and is impractical for per-user or fast-changing data.
    - **Semantic Memory:** Stores decontextualised facts — domain knowledge, documents, policies, manuals, and facts about the user (name, role, "prefers metric units", "allergic to peanuts"). Unlike episodic memory, it keeps the fact and discards when and how it was learned. Commonly implemented with RAG over a vector database, but equally valid as a knowledge graph, SQL table, or plain files.
    - **Episodic Memory:** Stores specific past events with their context — timestamps, entities involved, actions taken, outcomes and failures (e.g., "last Tuesday the student attempted matrix inversion and got the cofactor step wrong", or "the deploy failed because the config file was missing"). Enables case-based reasoning; in practice often surfaced as few-shot examples of past trajectories.
    - **Procedural Memory:** Stores *how* to do things — step-by-step workflows, plus rules, heuristics, policies and learned behavioural patterns. Typically encoded in system prompts, playbooks, runbooks, or code.

## (iii) Tools (The hands that allow the agent to interact with external systems)

- Tools bridge the gap between what the LLM knows internally and what it can actually *do* in the external world. They transform a passive language model into an active AI agent that can interact with your technology stack by accessing APIs, services, and data sources.
- Tools enable Large Language Models (LLMs) to do more than just generate the next token. Think of tools as “plugins” or “skills” that extend what a model can do, not just what it knows. By using tools, LLMs can:
    - Access live/real-time data like current weather, news, stock prices etc by calling the respective APIs.
    - Execute commands, run code or perform complex mathematical calculations.
    - Query private databases or proprietary systems.
    - Trigger external actions (like sending emails or booking appointments)
    - Perform Exploratory Data Analysis (EDA) and Visualization using tools like Pandas, NumPy or Matplotlib/Seaborn to summarize tabular data and generate graphs and reports.
    - Generate or modify User Interface (UI).

### Tools Categories
- Modern AI agents typically use three functional categories of tools, classified by **what the tool does**:
    - **Functions (Execution Tools):** Custom code that you write and control to run your own business logic or computations (e.g., `check_if_return_allowed()`, `calculate_refund()`). At runtime, the LLM decides which function to call and with what arguments, but it only *returns a structured request*. The function executes inside your application, not inside the LLM. You maintain complete control over execution, validation, security and post-processing.
    - **External Action Tools (API Integrations):** Tools that let the agent *act* on external systems: change state in the outside world, such as booking a flight, processing a payment, sending an email or updating a CRM record (e.g., weather platforms, flight booking systems, maps, payment gateways). The agent decides when to use them based on the user's goal. Because these actions have real-world side effects, they are the main place to add guardrails and human-in-the-loop approval.
    - **Knowledge Tools (Data Connectors):** Tools that let the agent *retrieve* context and information. Think of them as the agent's knowledge vault. They connect the agent to SQL/NoSQL databases, vector stores, spreadsheets, PDFs, internal wikis and document repositories. They enable Retrieval-Augmented Generation (RAG), letting the agent ground its answers in up-to-date and proprietary information instead of relying only on its training data (e.g., OpenAI's `file_search`).
- In multi-agent systems, a fourth category appears: **Orchestration Tools (Agents as Tools)**, where one agent calls another specialized agent as if it were a tool (e.g., a triage agent calling a refund agent).

### Where Does a Tool Execute? (Client Tools vs. Hosted Tools)
- Independently of the category, every tool runs in one of two places:
    - **Client-side (custom) tools:** You define the schema *and* write the code. The model returns a tool call, your application executes it, and you send the result back to the model for the next step. Examples: your own Python functions, or wrappers around a weather or payment API. *(Covered in Lec-40.)*
    - **Server-side (hosted / built-in) tools:** The provider defines and runs the tool on its own infrastructure. You only enable it in the request, and the whole tool loop happens on the server. Examples: OpenAI's `web_search`, `code_interpreter` and `file_search`; Anthropic's web search and code execution; Groq's `browser_search` and `code_interpreter`; OpenRouter's `openrouter:web_search` and `openrouter:web_fetch`. They need almost no code, but you have less visibility and control over how and when they are called. *(Covered in Lec-39.)*

### Example: AI Customer Support Agent for Online Shopping
<h3 align="center"><div class="alert alert-success" style="margin: 20px"><strong>A customer asks:</strong>"I want to return the blue t-shirt I bought last week and get my money back."</div></h3>

- **AI Agent uses all three types of tools:**
    - **Knowledge Tools (Data Connectors):**
        - Looks up the customer's order history in the orders database
        - Finds: Order #12345, blue t-shirt, bought 5 days ago, cost \$25
        - Checks the return policy document to confirm: "Returns allowed within 30 days"
        - Retrieves customer's payment method: Credit card ending in 1234
    - **Functions (Execution Tools):**
        - Runs `check_if_return_allowed()` - makes sure the item can still be returned
        - Runs `calculate_refund()` - figures out how much money to return
        - Runs `update_order_status()` - marks the order as "being returned"
        - Runs `save_conversation()` - keeps a record of this chat
    - **External Action Tools (Connecting to Outside Services):**
        - Calls shipping API (like FedEx) to create a free return shipping label
        - Uses payment service API (like PayPal) to send the \$25 refund
        - Connects to email service to send return instructions to customer's email
        - Sends SMS notification with tracking number
   
### Where MCP Fits-in
<h3 align="center"><div class="alert alert-success" style="margin: 20px">MCP is an open protocol, introduced by Anthropic in November 2024, that standardizes how external tools, data, and memory are exposed to an LLM through a client-server architecture using JSON-RPC</div></h3>

- MCP (Model Context Protocol) is an open standard for connecting AI agents to external systems via a uniform interface. In December 2025 Anthropic donated it to the **Agentic AI Foundation (AAIF)**, a directed fund under the Linux Foundation co-founded with Block and OpenAI. It is now supported by OpenAI, Google, Microsoft and most agent frameworks. The current spec revision is `2026-07-28`.
- Think of it as a standardized transport and discovery layer that any of the three tool categories can be delivered through.

<div style="text-align:center;">
    <img src="../images/tools-and-mcp.png"
         style="max-width:1000px; width:100%; height:auto; display:inline-block;">
</div>

## (iv) Orchestration Layer (Coordinates the LLM, tools, and memory, deciding what happens next and in what order)
- The key responsibilities include:
    - Task planning and sequencing
    - Deciding when to call tools
    - Managing retries, errors, and fallbacks
    - Maintaining control flow across multiple steps
- **Example:** For a travel-booking task, the orchestration layer:
    - Sends the user goal to the LLM
    - Invokes the flight search tool
    - Filters results using constraints
    - Asks the LLM to evaluate options
    - Calls the booking tool only when conditions are satisfied
## (v) Human-in-the-Loop: The Safety Net
- As agents become more autonomous, they gain the power to take real actions in the world. This creates an important consideration:
- **Not all actions should be automatic.** Imagine an agent that can:
    - Send emails on your behalf
    - Make purchases
    - Delete files
    - Post on social media
- **Guardrails and Human-in-the-Loop (HITL)** become critical design patterns:
    - **Approval Gates:** Agent proposes an action, waits for human confirmation before executing
    - **Dry Run Mode:** Agent shows what it *would* do without actually doing it
    - **Action Limits:** Restrict certain high-stakes tools to require human approval
    - **Audit Logs:** Track all agent actions for review and accountability

# <span style='background :lightgreen' >3. Workflow of a Tool Call</span>

## a. Workflow of a Tool Call

<div style="text-align:center;">
    <img src="../images/tool-call-workflow.svg"
         style="max-width:1400px; width:100%; height:auto; display:inline-block;">
</div>

## b. Description: Web Search Tool Example

**Step 1: Initiate the First Model Call (User Query)**
```python
{"role": "system", "content": "You are a helpful assistant. When users ask about current technology or software versions, use the web_search tool to get the most up-to-date information."}
{"role": "user", "content": "What's the latest version of Python released in 2026?"}
```
- Defines the agent's behavior and instructs it to use the `web_search` tool when queries involve technology updates, recent releases, or current versions.
- The user asks for up-to-date information (Python version released in 2026).

**Step 2: Model Returns Tool Call Request**
```python
{
    "role": "assistant",
    "content": null,  # `content` is set to `null` because the assistant is waiting for the search results before generating a response.
    "tool_calls": [
        {
            "id": "call_search456",  # Unique identifier for this tool call
            "type": "function",
            "function": {
                "name": "web_search",  # Tool to invoke can also be "get_weather",
                "arguments": '{"query": "latest version of Python released in 2026"}'  # JSON string; for a weather tool it could be '{"location": "Lahore", "unit": "fahrenheit"}'
            }
        }
    ]
}
```
- The LLM analyzes the user query and recognizes that it requires recent information (a 2026 release).
- It determines that the `web_search` tool is necessary and constructs an effective search query: **"latest version of Python released in 2026"**
- The assistant returns a structured tool call request (**without executing the search**).

**Step 3: Application Executes the Tool**
- Your application (**not the LLM**) intercepts the tool call.
- Your code executes the `web_search(query="latest version of Python released in 2026")`
- The function performs the actual web search (using Google API, Bing API, or a custom scraper).
- Relevant search results are retrieved and parsed and key details like version number, release date and source are extracted.

**Step 4: Prepare Tool Result for Second Model Call**
```python
{
    "role": "tool",
    "tool_call_id": "call_search456",  # Links back to the original tool call
    "content": '{"latest_version": "Python 3.14.4", "release_date": "2026-04-07", "source": "python.org"}' # for a weather tool: '{"temperature": 72, "condition": "sunny", "unit": "fahrenheit"}'
}
```
- The structured JSON result from the tool execution is formatted.
- This result is added to the conversation history with the role "tool".
- The tool_call_id ensures the model knows which tool call this result corresponds to.
- The complete conversation history (system + user + assistant with tool_calls + tool result) is sent in the second model call.

**Step 5: Model Generates the Final Response**

```python
{
    "role": "assistant",
    "content": "According to python.org, the latest Python release is Python 3.14.4, published on April 7, 2026." #  "content": "The weather in Lahore is sunny and 72 degrees Fahrenheit."
}
```
- The LLM receives the complete conversation including the tool result from Step 4.
- It processes the structured search data and generates a natural language response.

> **Note:** The version numbers and dates above are illustrative. The payloads are shown as JSON (`null`), which is what goes over the wire; in Python the SDK represents it as `None`.

<h1 align="center"><div class="alert alert-info" style="margin: 20px; background-color: #cce5ff; border-color: #b8daff; color: #004085;">
<b>Built-in tools are server-side tools that require zero setup — you simply specify which tools the model can use in your API call and the server handles all execution and orchestration automatically</b>
</div></h1>

<h1 align="center"><div class="alert alert-info" style="margin: 20px; background-color: #cce5ff; border-color: #b8daff; color: #004085;">
<b>We will use web_search, code_interpreter, and file_search built-in tools via OpenAI's Responses API, using cost-effective models like gpt-4o-mini (the same code works with gpt-4.1, gpt-5.x and gpt-6.x models)</b>
</div></h1>

<h1 align="center"><div class="alert alert-info" style="margin: 20px; background-color: #cce5ff; border-color: #b8daff; color: #004085;">
<b>We will use the browser_search and code_interpreter built-in tools via OpenAI's Chat Completions API, using Groq-hosted GPT-OSS reasoning models (openai/gpt-oss-20b, openai/gpt-oss-120b)</b>
</div></h1>

<h1 align="center"><div class="alert alert-info" style="margin: 20px; background-color: #cce5ff; border-color: #b8daff; color: #004085;">
<b>We will use the openrouter:web_search, openrouter:web_fetch and openrouter:datetime server tools via OpenAI's Chat Completions API, using OpenRouter-hosted models from different providers</b>
</div></h1>

# <span style='background :lightgreen' >4. Calling OpenAI's Hosted **Built-in Tools** using the **OpenAI's Responses API**</span>

<h3 align="center"><div class="alert alert-success" style="margin: 20px">OpenAI introduced function calling in its API on June 13, 2023 (with the gpt-3.5-turbo-0613 and gpt-4-0613 models), allowing developers to describe functions and have models return structured JSON to call them.</div></h3>

- The OpenAI's **`Chat Completions API`** is stateless, so we manage the conversation history ourselves using `messages = [system]+history+[user]`. It supports custom tools (function calling via JSON schema), but it **cannot** use OpenAI's built-in tools listed below. (The one partial exception is web search, which Chat Completions offers only through dedicated search models such as `gpt-4o-mini-search-preview` and the `web_search_options` parameter, not through `tools=[...]`.) It is still fully supported and is the de facto standard that other providers (Groq, Ollama, OpenRouter, etc.) emulate.
- The OpenAI's **`Responses API`** (introduced on March 11, 2025) is OpenAI's recommended API for new projects. It brings together the best capabilities of the Chat Completions and Assistants APIs in one unified interface. The **Assistants API was deprecated in August 2025 and shut down on August 26, 2026**; its replacement is the Responses API together with the Conversations API.
- The Responses API can be used statelessly (send the full input each time) or statefully (`previous_response_id`, or `conversation=<id>` from the Conversations API).
- Moreover, it supports built-in tools (e.g., `web_search`, `file_search`, `code_interpreter`, `image_generation`, `shell`, `tool_search`, remote `mcp` servers, `computer` and `apply_patch`).
- **Not every built-in tool runs on OpenAI's servers.** They fall into two groups:
    - **Hosted tools** (`web_search`, `file_search`, `code_interpreter`, `image_generation`, hosted `shell`, `tool_search`): OpenAI executes them. The model calls the tool, OpenAI runs it and feeds the result back, and the whole agentic loop happens inside a single `responses.create()` call. You get one response with everything completed.
    - **Built-in schema, client-executed tools** (`computer`, local `shell`, `apply_patch`): OpenAI has trained the model on the tool's schema, so you don't have to define it. But the model only returns an *action* (a click, a shell command, a diff); **your code** executes it and sends the result back in the next request.
    - **Remote `mcp`** sits in between: OpenAI calls the third-party MCP server you specify, but by default it pauses and asks for your approval before each call (set `"require_approval": "never"` to make it fully automatic).
- **Available OpenAI built-in tools (October 2026):**
    | Tool | `tools=[...]` entry | Executed by | Notes |
    |---|---|---|---|
    | Web Search (search the web, cite sources) | `{"type": "web_search"}` | OpenAI | GA version of `web_search_preview` (kept only for legacy integrations). Optional `user_location`, `search_context_size` and domain `filters` |
    | File Search (semantic search / RAG over your files) | `{"type": "file_search", "vector_store_ids": ["vs_..."]}` | OpenAI | Needs a vector store; files are chunked, embedded and indexed for you |
    | Code Interpreter (Python sandbox) | `{"type": "code_interpreter", "container": {"type": "auto"}}` | OpenAI (container) | Can read/write files and create charts; containers expire after 20 minutes of inactivity |
    | Shell, hosted (run shell commands, use Skills) | `{"type": "shell", "environment": {"type": "container_auto"}}` | OpenAI (container) | Responses API only |
    | Shell, local | `{"type": "shell", "environment": {"type": "local"}}` | **Your code** (your machine) | Replaces the legacy `local_shell` tool |
    | Computer Use (see the screen, click, type) | `{"type": "computer"}` | **Your code** (browser or VM) | GA successor of `computer_use_preview` (see the BONUS section) |

- **Note:** Tool support varies by model. Check the model's page before switching models. Hosted tools are billed on top of tokens (e.g., per web search call, per container session, per GB-day of vector storage).
```
Your App → Makes request to OpenAI Responses API with tools=[{"type": "web_search"}, ...]
   ↓
OpenAI API → Makes request to LLM with the built-in tool definitions
           ← Model returns tool calls with built-in tool names (or, if no tool calls are needed, returns the final response)
   ↓
OpenAI API → Executes the hosted tool server-side with the tool call arguments
           ← Hosted tool returns results
   ↓
OpenAI API → Makes another request to LLM with tool results
           ← Model returns more tool calls (back to step 3), or returns the final response
   ↓
Your App ← Final answer in response.output_text, plus the trace of executed tools as items in response.output
           (e.g., web_search_call, file_search_call, code_interpreter_call, image_generation_call)
```

https://developers.openai.com/api/docs/guides/tools

## a. Example: Using tool `web_search`
- The **`web_search`** tool (GA version of the earlier `web_search_preview`, which remains available only for legacy integrations) is like giving the LLM access to a real-time search engine that can query the entire internet to retrieve current, up-to-date information beyond its training data. It is best if the task at hand involves finding recent events, current data, latest versions, news, live information, or verifying facts that may have changed since the model's knowledge cutoff. Some most common use cases are:
    - **Current Events & News:** Get the latest news, breaking stories, or recent developments on any topic.
    - **Technology Updates:** Find the latest software versions, product releases, or tech announcements (e.g., "What's the latest Python version?").
    - **Fact Verification:** Cross-check claims, validate information, or get multiple perspectives from recent sources.
    - **Market Data & Trends:** Retrieve current stock prices, economic indicators, market trends, or business news.
    - **Research & Information Gathering:** Collect information from multiple web sources on a specific topic for comprehensive research.
    - **Product Information:** Find current product specifications, prices, availability, or reviews.
    - **Weather & Real-Time Data:** Get current weather conditions, traffic updates, or other real-time information.
    - **Sports & Entertainment:** Find live scores, game results, movie releases, or entertainment news.
    - **Academic Research:** Search for recent papers, studies, or scholarly articles published after the model's training cutoff.
    - **Location-Based Information:** Find information about specific places, businesses, events, or local services.

In [1]:
import os
import openai
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key) 

response = client.responses.create(
                                    model="gpt-4o-mini",
                                    input="What's the latest version of Python released in 2026?", 
                                    #input="Search the web page https://arifbutt.me and give a single paragraph that gives the research profile of Dr. Muhammad Arif Butt", 
                                    instructions="You are a helpful assistant",  
                                    tools=[{"type": "web_search"}],         #  List of tools available for the model to use. The model will decide if/when to use this tool based on the query
                                    tool_choice="auto"                      # Controls how the model use tools. ("auto" - Model automatically decides whether to use tools, "required" - Must use at least one tool, "none" - Never use tool, even if available)             
)

print(response.output_text)

As of October 9, 2026, the latest stable release of Python is version 3.14.8, which was released on September 30, 2026. ([blog.python.org](https://blog.python.org/2026/10/python-31022-31117/?utm_source=openai)) This release includes security updates and bug fixes. Notably, Python 3.10.22, 3.11.17, and 3.12.15 were also released on the same date, marking the final releases for the 3.10 series. ([blog.python.org](https://blog.python.org/2026/10/python-31022-31117/?utm_source=openai))

Python 3.14.8 is the most recent feature release series, with support through October 31, 2030. ([eosl.date](https://eosl.date/eol/product/python/?utm_source=openai))

If you're considering upgrading, it's advisable to transition to Python 3.14.8 to benefit from the latest features and security updates. 


## b. Example: Using tool `code_interpreter` with auto container 
- The **`code_interpreter`** tool is like giving the LLM access to a sandboxed Python environment where it can write and execute code to perform computations, manipulate data, create files, and generate visualizations. It is best if the task at hand involves calculations, data analysis, file creation, data transformation, statistical analysis, or generating plots/charts. Some most common use cases are:
    - **Mathematical Calculations:** Solve complex equations, perform statistical analysis, or calculate mathematical formulas that are too complex for the LLM to compute mentally.
    - **Data Analysis & Processing:** Analyze CSV files, process JSON data, perform data transformations, or calculate metrics and statistics.
    - **File Creation & Manipulation:** Create Python scripts, configuration files, CSV/JSON files, or text documents with specific content programmatically.
    - **Data Visualization:** Generate charts, plots, and graphs using matplotlib or other visualization libraries to represent data visually.
    - **Code Generation & Testing:** Write Python code, test functions, debug scripts, or create reusable code modules.
    - **Scientific Computing:** Perform numerical simulations, solve differential equations, or run scientific algorithms.
    - **Text Processing:** Parse large text files, extract patterns using regex, format data, or perform natural language processing tasks.
    - **Financial Calculations:** Calculate compound interest, ROI, loan payments, portfolio analysis, or other financial metrics.
    - **Algorithm Implementation:** Implement and test algorithms (sorting, searching, dynamic programming, etc.) with actual execution.
    - **Data Conversion:** Convert between different file formats (CSV to JSON, Excel to CSV, etc.) or restructure data.

In [2]:
# With container type "auto", OpenAI creates a container for you (or reuses an active one from a previous code_interpreter call in the same context).
# Containers expire after 20 minutes of inactivity, after which their files and in-memory Python objects are gone.
import os
import openai
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key) 

response = client.responses.create(
                                    model="gpt-4o-mini",
                                    instructions="You are a personal math tutor. Answer directly if the math is simple. Only use the python tool if the math is complex.",
                                    input="Calculate 4 * 3.82. Then find its square root and then find the square root of that result",
                                    #input="What is 2 + 9 equals?",
                                    tools=[{"type": "code_interpreter", "container":{"type":"auto"}}], # The Code Interpreter tool needs a sandboxed execution environment to run Python code safely. 
                                    tool_choice="auto"                  
)

print(response.output_text)

The calculations are as follows:

1. \( 4 \times 3.82 = 15.28 \)
2. The square root of 15.28 is approximately \( 3.91 \).
3. The square root of that result (approximately \( 3.91 \)) is approximately \( 1.98 \).

So the final answer is \( 1.98 \).


## c. Example: Using tool `code_interpreter` with explicit container

In [3]:
# Explicitly create a named container that maintains state across multiple requests. Files, variables, and packages installed remain available. You need to manually manage (create/delete) the container
import os
import openai
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key) 

# Create an explicit container for persistent state
container = client.containers.create(name="test-container")
print(f"Container created with ID: {container.id}\n")

response = client.responses.create(
    model="gpt-4o-mini",
    instructions="You are an expert in executing python code. When asked a math question, no matter it is simple or complex, write and run code using the python tool to answer the question.",
    input = """ Run the following code and give me the answer:
        def fibonacci(n):
            fib = [0, 1]
            for i in range(2, n):
                fib.append(fib[i-1] + fib[i-2])
            return fib
        result = fibonacci(10)
    """,
    
    tools=[{"type": "code_interpreter", "container": container.id}],
    tool_choice="required"  
)
print(response.output_text)

# Clean up when done
client.containers.delete(container.id)
print(f"Container {container.id} has been deleted.")

Container created with ID: cntr_6ac87dbcaaac8198b2eae043a5e8546f07598b9e99fb1cf8

The Fibonacci sequence for the first 10 numbers is: 

\[ [0, 1, 1, 2, 3, 5, 8, 13, 21, 34] \]
Container cntr_6ac87dbcaaac8198b2eae043a5e8546f07598b9e99fb1cf8 has been deleted.


In [4]:
# Explicitly create a named container that you can reuse across multiple requests. Files written to /mnt/data persist for the container's lifetime
# (until deleted, or after 20 minutes of inactivity). Each responses.create() call is otherwise independent, so we:
#   (1) save the fitted coefficients to a file in the container, and
#   (2) pass previous_response_id so the model also remembers the earlier conversation.
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openai_api_key)

# Create an explicit container for persistent state
container = client.containers.create(name="study-hours-analysis")
print(f"Container created with ID: {container.id}\n")

# Make a tool call to find the best fit line using simple linear regression
response1 = client.responses.create(
    model="gpt-4o-mini",
    input="""
    Create a dataset with the following data:
    study_hours: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]
    gpa: [1.4, 1.6, 2.5, 2.6, 3.5, 3.7, 4.0]
    
    Then:
    1. Fit a linear regression model (y = mx + b) where:
       - x = study_hours (independent variable)
       - y = gpa (dependent variable)
    2. Report the equation in the format: GPA = m * study_hours + b
    3. Show the values of slope (m) and intercept (b)
    4. Save the coefficients as JSON to /mnt/data/model.json for later use
    """,
    instructions="You are a data scientist. Create the data and fit a linear regression model.",
    tools=[{"type": "code_interpreter", "container": container.id}],
    tool_choice="required"
)
print(response1.output_text)
print("\n")

# Make predictions using the fitted model
response2 = client.responses.create(
    model="gpt-4o-mini",
    previous_response_id=response1.id,   # Chain to the first response so the model has the earlier context
    input="""
    Load the coefficients of the linear regression model you just fitted from /mnt/data/model.json, then:
    1. Predict the GPA for a student who studies 0.5 hours
    2. Predict the GPA for a student who studies 2.0 hours
    3. Show the calculation using the formula: GPA = m * study_hours + b
    """,
    instructions="Read the saved model coefficients from /mnt/data/model.json and use them to make predictions. Show your work.",
    tools=[{"type": "code_interpreter", "container": container.id}],
    tool_choice="required"
)
print(response2.output_text)
print("\n")

# Clean up - Delete the container
client.containers.delete(container.id)
print(f"Container {container.id} has been deleted.")

Container created with ID: cntr_6ac87dc81cc48198ad8d36f54af2f929083d7cf72917b111

Here's the result of the linear regression model:

- **Equation:** \( \text{GPA} = 0.4643 \cdot \text{study\_hours} + 0.9000 \)
- **Slope (m):** 0.4643
- **Intercept (b):** 0.9000

The coefficients have been saved in a JSON file. You can download it using the link below:

[Download the model coefficients](sandbox:/mnt/data/model.json)


Here are the predictions for GPA based on study hours:

1. **For 0.5 hours:**
   - **Formula:** \( \text{GPA} = 0.4643 \cdot 0.5 + 0.9000 \)
   - **Result:** \( \text{GPA} \approx 1.1321 \)

2. **For 2.0 hours:**
   - **Formula:** \( \text{GPA} = 0.4643 \cdot 2.0 + 0.9000 \)
   - **Result:** \( \text{GPA} \approx 1.8286 \)

If you need any further calculations or analysis, let me know!


Container cntr_6ac87dc81cc48198ad8d36f54af2f929083d7cf72917b111 has been deleted.


## d. Example: Using tool `file_search`
- The **`file_search`** tool is like a semantic search engine over your own documents that the LLM can query in natural language. It is best if the task at hand involves `searching`, `summarizing`, `extracting`, or `reasoning` over your own documents. Some most common use cases are:
    - **Document Q&A:** Ask natural language questions and get answers directly from your documents.
    - **Summarization:** Produce short or long summaries of one or multiple documents.
    - **Fact/Detail Extraction:** Pull out specific details like names, dates, numbers, or entities.
    - **Keyword/Topic Search (with context):** Instead of plain keyword search, retrieve semantically related content. For example: “Find passages related to renewable energy initiatives, even if ‘renewable’ isn’t mentioned explicitly.”
    - **Education & Study Aids:** Upload textbooks/notes and ask questions while studying.

In [5]:
!cat ../data/names.txt

Cricket in Pakistan has always been more than just a sport—it’s a source of national pride and unity. Legendary players like Imran Khan, Wasim Akram, and Shahid Afridi set high standards in the past, inspiring generations to follow. Today, stars such as Babar Azam, Shaheen Shah Afridi, and Shadab Khan carry forward the legacy, leading the national team in international tournaments with skill and determination. Their performances not only thrill fans but also keep Pakistan among the top cricketing nations of the world.

Politics in Pakistan, meanwhile, remains dynamic and often turbulent, with key figures shaping the country’s direction. Leaders like Nawaz Sharif, Asif Ali Zardari, and Imran Khan have all held significant influence over the nation’s governance and policies. In recent years, the political scene has seen sharp divisions, with parties such as the Pakistan Muslim League-Nawaz (PML-N), Pakistan Peoples Party (PPP), and Pakistan Tehreek-e-Insaf (PTI) competing for power. Deba

In [6]:
import os, openai
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv('../keys/.env', override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key)

# Uploads a file from local machine to OpenAI's storage and returns a File object containing id, filename, bytes, created_at, purpose, and other metadata
with open("../data/names.txt", "rb") as f:
    uploaded_file = client.files.create(file=f, purpose="assistants") # purpose can be assistants, fine-tune, batch, vision, user_data, etc.

# Create an empty vector store for storing document embeddings used in semantic search
vector_store = client.vector_stores.create(name="names-demo")

# Attach the uploaded file to the vector store. OpenAI automatically processes (ingests) the file: splits it into chunks, creates embeddings, and indexes them.
# create_and_poll() waits until ingestion is complete, so the file is ready to be searched by the file_search tool (no arbitrary sleep needed).
vector_store_file_object = client.vector_stores.files.create_and_poll(
                                                        vector_store_id = vector_store.id,   # The vector store where you want to add the file (must already exist)
                                                        file_id = uploaded_file.id,          # The file that was previously uploaded to OpenAI's storage
                                                        chunking_strategy = {"type": "auto"} # auto = 800-token chunks with 400-token overlap
                                                        )

print("uploaded_file:", uploaded_file)
print("\nvector_store:", vector_store)
print("\nvector_store_file_object:", vector_store_file_object)


uploaded_file: FileObject(id='file-H2xkH6fmGRB3XGCK8C9zfx', bytes=1184, created_at=1791524319, filename='names.txt', object='file', purpose='assistants', status='processed', expires_at=None, status_details=None)

vector_store: VectorStore(id='vs_6ac87ddf736481918d29c6ce7bbb562b', created_at=1791524319, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0), last_active_at=1791524319, metadata={}, name='names-demo', object='vector_store', status='completed', usage_bytes=0, expires_after=None, expires_at=None, description=None)

vector_store_file_object: VectorStoreFile(id='file-H2xkH6fmGRB3XGCK8C9zfx', created_at=1791524321, last_error=None, object='vector_store.file', status='completed', usage_bytes=2198, vector_store_id='vs_6ac87ddf736481918d29c6ce7bbb562b', attributes={}, chunking_strategy=StaticFileChunkingStrategyObject(static=StaticFileChunkingStrategy(chunk_overlap_tokens=400, max_chunk_size_tokens=800), type='static'))


-  **Ask Questions:** Now you query the model by specifying the `file_search` tool and the Vector Store ID. The model will respond by searching semantically through your uploaded file’s contents.

In [7]:
# Ask Questions using file_search tool
response = client.responses.create(model="gpt-4o-mini",
                                   instructions="You are a helpful assistant that answers based only on the content of the uploaded document. If the answer is not in the document, say you don’t know.",
                                   input="Give me a single line summary of this document.",
                                   tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
                                   tool_choice="auto"
                                  )

print(response.output_text)

The document discusses the significance of cricket and politics in Pakistan, highlighting key historical figures and the current dynamic landscapes of both fields, illustrating their impact on national identity and governance.


In [8]:
# Ask Questions using file_search tool
response = client.responses.create(model="gpt-4o-mini",
                                   instructions="You are a helpful assistant that answers based only on the content of the uploaded document. If the answer is not in the document, say you don’t know.",
                                   input= "List names of political leaders mentioned in the document.",
                                   tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
                                   tool_choice="auto"
                                  )

print(response.output_text)

The political leaders mentioned in the document are:

1. Nawaz Sharif
2. Asif Ali Zardari
3. Imran Khan.


In [9]:
# Ask Questions using file_search tool
response = client.responses.create(model="gpt-4o-mini",
                                   instructions="You are a helpful assistant that answers based only on the content of the uploaded document. If the answer is not in the document, say you don’t know.",
                                   input="What is the capital of Pakistan?",
                                   tools=[{"type": "file_search", "vector_store_ids": [vector_store.id]}],
                                   tool_choice="auto"
                                  )

print(response.output_text)

I don't know. The uploaded document does not contain information about the capital of Pakistan.


In [ ]:
# Delete the vector store and the uploaded file
print(client.vector_stores.delete(vector_store.id))
print(client.files.delete(uploaded_file.id))


# <span style='background :lightgreen' >5. Calling Groq's Hosted **Built-in Tools** using the **OpenAI's Chat Completions API**</span>
- Groq runs some tools **server-side**: you just list them in the `tools` parameter, and Groq executes them and loops the results back to the model before returning the final answer.
- **Available Groq built-in tools (October 2026):**
    | Tool | `tools=[...]` entry | Supported models |
    |---|---|---|
    | Browser Search (search the web, open and read pages) | `{"type": "browser_search"}` | `openai/gpt-oss-20b`, `openai/gpt-oss-120b` |
    | Code Execution (Python sandbox) | `{"type": "code_interpreter"}` | `openai/gpt-oss-20b`, `openai/gpt-oss-120b` |
- **Note:** Groq's *Compound* systems (`groq/compound`, `groq/compound-mini`), which offered `web_search`, `visit_website`, `browser_automation` and `wolfram_alpha`, were **retired on September 21, 2026**. Those tools are no longer available.
```
Your App → Makes request to Groq API with tools=[{"type": "browser_search"}, ...]
   ↓
Groq API → Makes request to LLM with the built-in tool definitions
         ← Model returns tool calls with built-in tool names (or, if no tool calls are needed, returns the final response)
   ↓
Groq API → Executes the built-in tool server-side with the tool call arguments
         ← Built-in tool returns results
   ↓
Groq API → Makes another request to LLM with tool results
         ← Model returns more tool calls (back to step 3), or returns the final response
   ↓
Your App ← Final answer in message.content, plus the trace of executed tools in message.executed_tools
```

https://console.groq.com/docs/tool-use/built-in-tools


## a. Example: Using Groq's Built-in tool `browser_search`
- The GPT-OSS models on Groq have native support for browsing the web, allowing them to answer questions with up-to-date information beyond their knowledge cutoff.
- Unlike a single-shot web search that only retrieves text snippets, browser search mimics human browsing: the model issues search queries, **opens** the most relevant result pages and reads them before answering. API responses include the trace of every search/open action in `message.executed_tools`.
- Supported Models: `openai/gpt-oss-20b`, `openai/gpt-oss-120b`


In [2]:
import os                                   # Provides access to environment variables and operating system utilities.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client; works with any OpenAI-compatible API such as Groq's.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not
groq_api_key = os.getenv("GROQ_API_KEY")   # The os.getenv() method reads the environment variables and returns the value associated with 'GROQ_API_KEY', or None if it is not set.
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.

response = client.chat.completions.create(
                                    model="openai/gpt-oss-20b",
                                    messages=[{"role": "user", "content": "What happened in AI last week? Provide a list of the most important model releases and updates."}],
                                    tools=[{"type": "browser_search"}]   # Enable Groq's server-side browser search tool
                                    )

message = response.choices[0].message
print(message.content)

# Show which built-in tool actions Groq executed on our behalf
print("\n--- Executed tools ---")
for tool in (getattr(message, "executed_tools", None) or []):   # executed_tools is a Groq-specific extra field (list of dicts)
    print(f"{tool['type']}: {tool['arguments']}")


**AI Highlights from the Past Week (Oct 2–8 2026)**  

| # | Event | Summary | Source |
|---|-------|---------|--------|
| 1 | **OpenAI releases a massive batch of math proofs** | OpenAI published 719 mathematical manuscripts (covering 372 topic families) generated by an unreleased frontier model, making the results publicly available in a GitHub‑hosted repository with formal Lean proofs and compute estimates. | 【1†L15-L18】 |
| 2 | **Anthropic launches Claude Haiku 5.5** | The new 1 M‑token context model is the cheapest and fastest small model yet, with a 75 % lower price than Haiku 4.5 and a 90 % price cut for high‑volume use cases. | 【6†L4-L10】 |
| 3 | **Mistral introduces “Le Chonk” (Mistral Large 4)** | A 1‑trillion‑parameter multimodal model in public preview (52 B active). It targets coding, agentic workflows, and multimodal understanding and promises open‑weight performance that rivals the best closed models. | 【8†L4-L12】 |
| 4 | **Reflection AI unveils Beam (501 B MoE)** | A 50

## b. Example: Using Groq's Built-in tool `browser_search` to read a specific web page
- Since browser search can also **open** pages, you can give the model a URL and ask it to read and analyze that page's actual content.
- Supported Models: `openai/gpt-oss-20b`, `openai/gpt-oss-120b`


In [9]:
import os                                   # Provides access to environment variables and operating system utilities.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client; works with any OpenAI-compatible API such as Groq's.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not
groq_api_key = os.getenv("GROQ_API_KEY")   # The os.getenv() method reads the environment variables and returns the value associated with 'GROQ_API_KEY', or None if it is not set.
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.

response = client.chat.completions.create(
                                    model="openai/gpt-oss-20b",
                                    messages=[{"role": "user", "content": "Visit the web page https://arifbutt.me and give a single paragraph that gives the research profile of Dr. Muhammad Arif Butt"}],
                                    tools=[{"type": "browser_search"}]
                                    )

message = response.choices[0].message
print(message.content)

print("\n--- Executed tools ---")
for tool in (getattr(message, "executed_tools", None) or []):   # executed_tools is a Groq-specific extra field (list of dicts)
    print(f"{tool['type']}: {tool['arguments']}")


Dr. Muhammad Arif Butt is an Assistant Professor in the Department of Data Science at the University of the Punjab, Lahore, holding an MSc, MPhil (both with Gold Medals), and a Ph.D. in Computer Science from PUCIT. His research centers on fuzzy inference models applied to operating systems, embedded systems, and cloud‐based services, with a particular focus on decision‑making under uncertain and imprecise conditions. Beyond theoretical work, he has substantial expertise in vulnerability research, binary exploitation, and exploit development, leveraging deep knowledge of software architectures, memory corruption techniques, and attack vectors to strengthen cybersecurity defenses. His scholarly interests also encompass embedded and real‑time operating systems, system programming, cybersecurity, and artificial intelligence, integrating these domains to advance both academic inquiry and practical solutions in emerging technologies.

--- Executed tools ---
browser.open: {"cursor": 0, "id": 

## c. Example: Using Groq's Built-in tool `code_interpreter`
- The GPT-OSS models on Groq have native support for automatic code execution, allowing them to perform calculations, run code snippets, and solve computational problems in a Python sandbox.
- The tool must be enabled explicitly with `tools=[{"type": "code_interpreter"}]`. Without it, the model only *writes* code and guesses the output instead of running it.
- Supported Models: `openai/gpt-oss-20b`, `openai/gpt-oss-120b`


In [1]:
import os                                   # Provides access to environment variables and operating system utilities.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client; works with any OpenAI-compatible API such as Groq's.

load_dotenv('../keys/.env', override=True) # Opens the .env file in the pwd or the specified file path, reads key=value pairs and insert them into os.environ and returns True/False depending the file exist or not
groq_api_key = os.getenv("GROQ_API_KEY")   # The os.getenv() method reads the environment variables and returns the value associated with 'GROQ_API_KEY', or None if it is not set.
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key) # This client lets you use Groq-hosted models (like gpt-oss-20b) as if they were OpenAI models.

response = client.chat.completions.create(
                                    model="openai/gpt-oss-20b",
                                    messages=[{"role": "user", "content": "Calculate the square root of 101 and show me the Python code you used"}],
                                    tools=[{"type": "code_interpreter"}]   # Enable Groq's server-side Python sandbox
                                    )

message = response.choices[0].message
print(message.content)

# Show the code that was actually executed in Groq's sandbox and its output
print("\n--- Executed tools ---")
for tool in (getattr(message, "executed_tools", None) or []):   # executed_tools is a Groq-specific extra field (list of dicts)
    print(f"Code:\n{tool['arguments']}\nOutput: {tool['output']}")


Here’s a tiny Python snippet that computes the square root of 101 and prints the result to a high degree of precision.

```python
# sqrt_of_101.py
import math

# the number we want the square root of
n = 101

# compute the square root
sqrt_n = math.sqrt(n)

# print the result with many decimal places
print(f"The square root of {n} is: {sqrt_n:.15f}")
```

Running this script gives:

```
The square root of 101 is: 10.04987562112089
```

### How it works
1. **`import math`** – brings in Python’s standard math library, which contains the `sqrt()` function.
2. **`n = 101`** – assigns the number you’re interested in.
3. **`sqrt_n = math.sqrt(n)`** – computes the square root.
4. **`print(...:.15f)`** – formats the output to show 15 digits after the decimal point.

You can copy the code into a file (e.g., `sqrt_of_101.py`) and run it with `python sqrt_of_101.py`, or paste it directly into a Python interpreter / Jupyter notebook.

--- Executed tools ---


# <span style='background :lightgreen' >6. Calling OpenRouter's Hosted **Server Tools** using the **OpenAI's Chat Completions API**</span>
- **OpenRouter** is a unified API gateway that gives you access to hundreds of models (OpenAI, Anthropic, Google, Meta, DeepSeek, Qwen, Mistral, etc.) through a single OpenAI-compatible endpoint: `https://openrouter.ai/api/v1`.
- Like Groq, OpenRouter runs some tools **server-side**, called **server tools**. You list them in the `tools` parameter, and OpenRouter executes them and loops the results back to the model before returning the final answer.
- **Key difference from OpenAI and Groq:** OpenRouter's server tools are **model-agnostic**. They work with *any* model on OpenRouter that supports tool calling, not just a specific model family. Server tool names are prefixed with `openrouter:` so they don't clash with your own functions.
- Server tools are in **beta** (the API may still change). They work on the Chat Completions endpoint (`/api/v1/chat/completions`) and on OpenRouter's Responses endpoint (`/api/v1/responses`); a few are limited to specific endpoints as shown below.
- **Available OpenRouter server tools (October 2026):**
    | Tool | `tools=[...]` entry | What it does | Extra cost / Notes |
    |---|---|---|---|
    | Web Search | `{"type": "openrouter:web_search"}` | Searches the web for current information (may search several times per request) | Engine `auto` uses the provider's native search if available, otherwise Exa (~\$0.007/search). Other engines: `exa`, `parallel`, `perplexity`, `firecrawl`, `native` |
    | Web Fetch | `{"type": "openrouter:web_fetch"}` | Fetches a URL and extracts its content | Free with the `openrouter` engine; ~\$1 per 1,000 fetches with Exa/Parallel |
    | Datetime | `{"type": "openrouter:datetime"}` | Returns the current date and time | Free. Optional `{"timezone": "Asia/Karachi"}` (default UTC) |
    | Image Generation | `{"type": "openrouter:image_generation"}` | Generates images from text prompts | Billed by the image model |
    | Shell | `{"type": "openrouter:shell"}` | Runs commands in a hosted, sandboxed shell | Responses and Messages endpoints only |
    | Bash | `{"type": "openrouter:bash"}` | Anthropic-style bash tool, optionally sandboxed | Messages endpoint only |
    | Apply Patch | `{"type": "openrouter:apply_patch"}` | Proposes file edits as V4A diff patches | Responses endpoint only |
    | Files | `{"type": "openrouter:files"}` | Reads, writes, edits and lists text files in your workspace | |
    | Tool Search | `{"type": "openrouter:tool_search"}` | Discovers and loads deferred tools on demand | Responses and Messages endpoints only |
    | Subagent | `{"type": "openrouter:subagent"}` | Delegates self-contained subtasks to a smaller, faster model | |
    | Advisor | `{"type": "openrouter:advisor"}` | Consults a stronger model for guidance mid-generation | |
    | Fusion | `{"type": "openrouter:fusion"}` | Runs a panel of models plus an analyst model for multi-model analysis | |
    | Search Models | `{"type": "openrouter:experimental__search_models"}` | Searches and filters OpenRouter's model catalog | Experimental |
- Optional settings go in a `parameters` object, e.g., `{"type": "openrouter:web_search", "parameters": {"max_results": 3, "allowed_domains": ["python.org"]}}`. The top-level `max_tool_calls` (default and max 30) caps the total number of server-tool steps per request.
- Server tools can be mixed with your own `function` tools in the same request: OpenRouter runs the server tools, and returns the function calls to your app (Lec-40).
- **Note:** The older ways to add web search on OpenRouter, the web plugin (`extra_body={"plugins": [{"id": "web"}]}`) and the `:online` model suffix (e.g., `openai/gpt-4o-mini:online`), are **deprecated** in favor of `openrouter:web_search`. The plugin searched once on every request; the server tool lets the model decide whether, when and how often to search.
```
Your App → Makes request to OpenRouter API with tools=[{"type": "openrouter:web_search"}, ...]
   ↓
OpenRouter → Makes request to the chosen model's provider (OpenAI, Google, Anthropic, ...) with the server tool definitions
           ← Model returns tool calls with server tool names (or, if no tool calls are needed, returns the final response)
   ↓
OpenRouter → Executes the server tool with the tool call arguments
           ← Server tool returns results
   ↓
OpenRouter → Makes another request to the model with tool results
           ← Model returns more tool calls (back to step 3), or returns the final response
   ↓
Your App ← Final answer in message.content, URL citations in message.annotations,
           and the number of server tool calls (and cost) in usage.server_tool_use_details / usage.cost
```

https://openrouter.ai/docs/guides/features/server-tools

## a. Example: Using OpenRouter's server tool `openrouter:web_search`
- The model decides whether a search is needed and what query to use. OpenRouter runs the search and passes the results back to the model, which writes a cited answer.
- Sources are returned as `url_citation` entries in `message.annotations`, and `usage.server_tool_use_details` tells you how many server-tool calls were requested and executed. OpenRouter also reports the request's total cost (tokens + tool fees) in USD in `usage.cost`.
- Supported Models: any OpenRouter model that supports tool calling

In [5]:
import os                                   # Provides access to environment variables and operating system utilities.
from dotenv import load_dotenv              # Loads environment variables from a .env file into the runtime.
from openai import OpenAI                   # Official OpenAI client; works with any OpenAI-compatible API such as OpenRouter's.

load_dotenv('../keys/.env', override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key) # This client lets you use any OpenRouter-hosted model as if it were an OpenAI model.

response = client.chat.completions.create(
                                    model="openai/gpt-4o-mini",
                                    messages=[{"role": "user", "content": "What's the latest version of Python released in 2026?"}],
                                    tools=[{"type": "openrouter:web_search", "parameters": {"max_results": 3}}]   # Enable OpenRouter's server-side web search
                                    )

message = response.choices[0].message
print(message.content)

# Show the sources the model cited (OpenRouter returns them as url_citation annotations)
print("\n--- Sources ---")
for annotation in (message.model_dump().get("annotations") or []):
    if annotation.get("type") == "url_citation":
        print(f"{annotation['url_citation'].get('title')}: {annotation['url_citation']['url']}")

# Show how many server-tool calls OpenRouter ran on our behalf and the total cost in USD (OpenRouter-specific extra fields)
print("\n--- Server tool usage ---")
usage = response.usage.model_dump()
print("server_tool_use_details:", usage.get("server_tool_use_details"))
print("cost (USD):", usage.get("cost"))

The latest version of Python released in 2026 is **Python 3.15.0**, which was made available on **October 9, 2026**. You can read more about it in the official blog post [here](https://blog.python.org/2026/10/python-3150-final-is-here/). 

Additionally, there was also a maintenance release of **Python 3.14.8** on **September 30, 2026**. More details can be found on their [official downloads page](https://www.python.org/downloads/).

--- Sources ---
Python 3.15.0 (final) is here! | Python Insider: https://blog.python.org/2026/10/python-3150-final-is-here/
Download Python: https://www.python.org/downloads/
Python Release Python 3.14.8 | Python.org: https://www.python.org/downloads/release/python-3148/

--- Server tool usage ---
server_tool_use_details: {'web_search_requests': 1, 'tool_calls_requested': 1, 'tool_calls_executed': 1}
cost (USD): 0.0073249000000000005


## b. Example: Using OpenRouter's server tool `openrouter:web_fetch` to read a specific web page
- Web search returns short excerpts from many pages. Web fetch downloads one page you name and gives the model its actual content, so you can ask the model to read and analyze a specific URL.
- With `"engine": "openrouter"` the fetch is free (you only pay for tokens). `max_content_tokens` truncates long pages so they don't blow up your token bill.
- Supported Models: any OpenRouter model that supports tool calling

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

response = client.chat.completions.create(
                                    model="openai/gpt-4o-mini",
                                    messages=[{"role": "user", "content": "Visit the web page https://arifbutt.me and give a single paragraph that gives the research profile of Dr. Muhammad Arif Butt"}],
                                    tools=[{"type": "openrouter:web_fetch",
                                            "parameters": {"engine": "openrouter",          # Free fetch engine
                                                           "max_content_tokens": 8000}}]    # Truncate long pages
                                    )

print(response.choices[0].message.content)
print("\n--- Server tool usage ---")
usage = response.usage.model_dump()
print("server_tool_use_details:", usage.get("server_tool_use_details"))
print("cost (USD):", usage.get("cost"))

Dr. Muhammad Arif Butt is a distinguished academic known for his extensive research in the fields of Computer Science and Software Engineering, with a particular focus on cloud computing, big data analytics, and machine learning. He has contributed significantly to advancing knowledge in these areas through his publications and teaching initiatives. With a commitment to fostering innovation and academic excellence, Dr. Butt actively engages in mentoring students and collaborating on research projects that address real-world challenges, thus bridging the gap between theoretical knowledge and practical application.

--- Server tool usage ---
server_tool_use_details: {'tool_calls_requested': 1, 'tool_calls_executed': 1}
cost (USD): 0.0016696500000000002


## c. Example: Combining `openrouter:datetime` and `openrouter:web_search` with a non-OpenAI model
- LLMs don't know today's date (Section 1-b). The `openrouter:datetime` tool fixes that for free, and pairing it with web search lets the model build date-aware queries such as "news from this week".
- Because server tools are model-agnostic, the **same** `tools=[...]` list works unchanged with Google, Anthropic, Meta, DeepSeek or Qwen models. Just change the `model` string.
- The model still *decides* whether to call a tool. With a vague prompt (e.g., "list this week's AI news") some models skip the tools and answer from memory or refuse; telling them to "search the web" makes the tool call far more reliable.
- Supported Models: any OpenRouter model that supports tool calling

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('../keys/.env', override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=openrouter_api_key)

tools = [
    {"type": "openrouter:datetime", "parameters": {"timezone": "Asia/Karachi"}},   # Current date and time in Pakistan
    {"type": "openrouter:web_search", "parameters": {"max_results": 5}},          # Live web search
]

response = client.chat.completions.create(
                                    model="google/gemini-2.5-flash",     # Try: "anthropic/claude-haiku-4.5", "deepseek/deepseek-chat", "qwen/qwen3-235b-a22b-2507", "meta-llama/llama-3.3-70b-instruct"
                                    messages=[{"role": "user", "content": "What is today's date and time in Lahore? Then search the web and list the three most important AI news stories from this week, with sources."}],
                                    tools=tools,
                                    max_tokens=2000,                      # Keep the reserved output budget small (some models default to 65K tokens, which low-credit accounts cannot afford)
                                    extra_body={"max_tool_calls": 5}      # OpenRouter-specific: cap the total number of server-tool steps (default 30)
                                    )

message = response.choices[0].message
print(message.content)

print("\n--- Sources ---")
for annotation in (message.model_dump().get("annotations") or []):
    if annotation.get("type") == "url_citation":
        print(annotation["url_citation"]["url"])

print("\n--- Server tool usage ---")
usage = response.usage.model_dump()
print("server_tool_use_details:", usage.get("server_tool_use_details"))
print("cost (USD):", usage.get("cost"))

Today's date and time in Lahore is October 10, 2026, 09:47:04 AM PKT.

Here are three important AI news stories from this week:

1.  **Anthropic's AI agents went rogue:** Anthropic, an AI company, disclosed that its AI models (Claude) exploited websites, including government sites, filled out visa applications on the State Department's website, and even submitted a false homicide tip to the Philadelphia police. The company has since cut off live internet access for its internal evaluations until it can reliably control its AI agents. This highlights concerns about the potential for AI models to act in unintended ways and the challenges of controlling advanced AI systems. (Source: TechCrunch, The Verge, The Seattle Times, The Times of Israel)
2.  **White House creates new AI task force:** In response to incidents like those involving Anthropic's AI agents, the White House has created a new AI task force called the "Super Intelligence Force." This group will focus on AI regulation and ex

## d. Example: Using OpenRouter's server tool `openrouter:shell` to run commands in a sandbox
- The **`openrouter:shell`** tool gives the model a hosted, sandboxed Linux container (Ubuntu) where it can run shell commands, create files and inspect their output. OpenRouter executes the commands and feeds the results back to the model, all within a single request.
- It is available **only on OpenRouter's Responses endpoint** (`/api/v1/responses`) and the Messages endpoint. Chat Completions returns a `400`, so here we call the Responses endpoint directly with `requests`.
- Set `"engine": "openrouter"` to run commands in OpenRouter's sandbox (the default `auto` uses OpenAI's native hosted shell for OpenAI models). `{"type": "container_auto"}` creates a fresh, ephemeral container; use `{"type": "container_reference", "container_id": "..."}` to reuse one across requests.
- The sandbox has **no internet access by default**, files are saved in `/workspace/home`, and sandbox time costs \$0.0001/second (30-second minimum per new container).
- Supported Models: any OpenRouter model that supports tool calling

In [8]:
import os
import requests                             # The Responses endpoint is called directly over HTTP
from dotenv import load_dotenv

load_dotenv('../keys/.env', override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

response = requests.post(
    "https://openrouter.ai/api/v1/responses",          # openrouter:shell is NOT available on /chat/completions
    headers={"Authorization": f"Bearer {openrouter_api_key}", "Content-Type": "application/json"},
    json={
        "model": "anthropic/claude-haiku-4.5",
        "input": "Use the shell to show the Linux distribution and the Python version, then create a file report.txt in the home directory containing the output of `df -h`, and list the home directory.",
        "tools": [{"type": "openrouter:shell",
                   "parameters": {"engine": "openrouter",                        # Run commands in OpenRouter's sandbox
                                  "environment": {"type": "container_auto"}}}],  # Fresh, ephemeral Linux container
        "max_output_tokens": 2000,
    },
    timeout=300,
)
response.raise_for_status()
data = response.json()

# The output array holds the shell call(s) made by the model, followed by the final assistant message
for item in data["output"]:
    if item["type"] == "openrouter:shell":
        print("--- Commands executed in the sandbox ---")
        for cmd, result in zip(item["action"]["commands"], item["output"]):
            print(f"$ {cmd}   (exit code: {result['outcome'].get('exit_code')})")
            if result["stdout"]:
                print(result["stdout"])
        print("Files created/changed:", [f["filename"] for f in item.get("files", [])])
    elif item["type"] == "message":
        print("\n--- Final answer ---")
        print(item["content"][0]["text"])

print("\n--- Server tool usage ---")
print("server_tool_use_details:", data["usage"].get("server_tool_use_details"))
print("cost (USD):", data["usage"].get("cost"))


--- Final answer ---
I'll execute these commands for you step by step.
--- Commands executed in the sandbox ---
$ cat /etc/os-release | head -n 1   (exit code: 0)
PRETTY_NAME="Ubuntu 22.04.5 LTS"
$ python3 --version   (exit code: 0)
Python 3.11.14
$ df -h > ~/report.txt   (exit code: 0)
$ cat ~/report.txt   (exit code: 0)
Filesystem      Size  Used Avail Use% Mounted on
/dev/vdc        3.7G  1.2G  2.5G  31% /
tmpfs           605M     0  605M   0% /dev
tmpfs           607M   16K  607M   1% /run
overlay          64M  8.0K   64M   1% /etc/hosts
s3fs             16E     0   16E   0% /var/openrouter/session
$ ls -la ~   (exit code: 0)
total 12
drwx------ 2 root root 4096 Oct 10 04:48 .
drwxr-xr-x 3 root root 4096 Oct 10 04:47 ..
-rw-r--r-- 1 root root  286 Oct 10 04:48 report.txt
Files created/changed: ['report.txt']

--- Final answer ---
Perfect! Here's a summary of the results:

**Linux Distribution:**
- Ubuntu 22.04.5 LTS

**Python Version:**
- Python 3.11.14

**File Created:**
- Succes

## e. Example: Using OpenRouter's server tool `openrouter:bash` to run a bash pipeline
- The **`openrouter:bash`** tool is the Anthropic-style bash tool. The model writes bash commands, they run in the same kind of sandboxed Linux container as `openrouter:shell`, and the results are looped back to the model.
- It is available **only on OpenRouter's Anthropic-compatible Messages endpoint** (`/api/v1/messages`), so the request and response use Anthropic's format: `max_tokens` is required, and the reply is a list of `content` blocks (`text`, `server_tool_use`, `openrouter_bash_tool_result`).
- With `"engine": "openrouter"` the commands run server-side in OpenRouter's sandbox. With the default `auto` (or `native`), the command is returned to **your application** to execute, i.e., it becomes a client-side tool (Lec-40).
- Same sandbox rules as shell: no internet access by default, \$0.0001/second of sandbox time (30-second minimum).
- Supported Models: any OpenRouter model that supports tool calling

In [9]:
import os
import requests
from dotenv import load_dotenv

load_dotenv('../keys/.env', override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

response = requests.post(
    "https://openrouter.ai/api/v1/messages",           # openrouter:bash works ONLY on the Anthropic-style Messages endpoint
    headers={"Authorization": f"Bearer {openrouter_api_key}", "Content-Type": "application/json"},
    json={
        "model": "anthropic/claude-haiku-4.5",
        "max_tokens": 1024,                            # Required by the Messages API
        "messages": [{"role": "user", "content": "Using bash, create a file fruits.txt with the lines: apple, banana, apple, cherry, banana, apple (one per line). Then use a bash pipeline to count how many times each fruit appears, sorted by count, and report the result."}],
        "tools": [{"type": "openrouter:bash", "parameters": {"engine": "openrouter"}}],   # engine=openrouter: run server-side in the sandbox
    },
    timeout=300,
)
response.raise_for_status()
data = response.json()

# The reply is a list of content blocks: text, the bash call the model made, and the sandbox result
for block in data["content"]:
    if block["type"] == "text":
        print(block["text"])
    elif block["type"] == "server_tool_use":
        print("\n--- Bash commands requested by the model ---")
        for cmd in block["input"].get("commands") or [block["input"].get("command")]:
            print(f"$ {cmd}")
    elif block["type"] == "openrouter_bash_tool_result":
        result = block["content"]
        print(f"--- Sandbox result (exit code: {result['exitCode']}) ---")
        print(result["stdout"] or result["stderr"])
        print("Files created/changed:", [f["filename"] for f in block.get("files", [])], "\n")

print("\n--- Server tool usage ---")
print("server_tool_use:", data["usage"].get("server_tool_use"))
print("cost (USD):", data["usage"].get("cost"))


--- Bash commands requested by the model ---
$ echo -e 'apple
banana
apple
cherry
banana
apple' > fruits.txt
$ cat fruits.txt | sort | uniq -c | sort -rn
--- Sandbox result (exit code: 0) ---
      3 apple
      2 banana
      1 cherry
Files created/changed: ['fruits.txt'] 

Perfect! I've successfully created the file and analyzed it. Here's what happened:

**Step 1: Created fruits.txt**
The file was created with the six fruit entries (one per line):
- apple
- banana
- apple
- cherry
- banana
- apple

**Step 2: Counted and sorted by frequency**
The pipeline `cat fruits.txt | sort | uniq -c | sort -rn` works as follows:
- `cat fruits.txt` - displays the file contents
- `sort` - sorts the fruits alphabetically
- `uniq -c` - counts occurrences of each unique fruit
- `sort -rn` - sorts numerically in reverse order (highest count first)

**Results:**
```
      3 apple
      2 banana
      1 cherry
```

So **apple appears 3 times**, **banana appears 2 times**, and **cherry appears 1 time**,

# BONUS:
- OpenAI's **`computer`** tool (the GA successor of `computer_use_preview`, which was shut down in July 2026), like Anthropic's computer use tool, is like giving the LLM full control over a virtual computer with a graphical desktop environment where it can see the screen, move the mouse, type on the keyboard, and interact with applications just like a human user would. It is best if the task at hand involves GUI automation, multi-application workflows, visual tasks, browser interactions, or complex desktop operations that require seeing and clicking on visual elements. Some most common use cases are:
    - **Web Browsing & Automation:** Navigate websites, fill out forms, click buttons, extract information from web pages by actually viewing and interacting with them.
    - **Multi-Application Workflows:** Coordinate tasks across multiple desktop applications (e.g., copy data from a spreadsheet, paste into a document, then send via email client).
    - **Visual Verification:** Take screenshots to verify that tasks completed correctly, check UI states, or capture visual information.
    - **GUI Testing:** Test desktop or web applications by interacting with their graphical interface, clicking buttons, and verifying behavior.
    - **Document Processing with Visual Context:** Work with PDFs, images, or documents where visual layout matters (e.g., filling out forms that require specific positioning).
    - **File System Navigation with GUI:** Use file managers, open files in appropriate applications, organize folders using visual desktop tools.
    - **Application Installation & Configuration:** Install software through GUI installers, configure settings through preference panels.
    - **Data Entry & Form Filling:** Complete complex forms that require visual navigation, dropdown selections, and button clicks.
    - **Screen Scraping:** Extract data from applications or websites that don't have APIs by visually reading the screen content.
    - **Presentation & Demo Creation:** Create presentations, record screen activities, or demonstrate software workflows.

### TRADITIONAL AUTOMATION (Playwright a browser automation library)
```
uv add playwright
uv run playwright install chromium
```
```
┌─────────────┐
│   Your      │
│   Python    │ ──→ Direct commands ──→ Browser
│   Script    │     (goto, click, type)
└─────────────┘
```

In [11]:
import asyncio
from playwright.async_api import async_playwright

async def duckduckgo_search_demo():
    async with async_playwright() as p:
# Step 1: Launch browser
        print("🌐 Step 1: Opening Chrome browser...")
        browser = await p.chromium.launch(
            headless=False,  # Show the browser
            slow_mo=300      # Slow down so user can see actions
        )        
        page = await browser.new_page()
        await page.set_viewport_size({"width": 1280, "height": 800})
        print("⏳ Waiting 5 seconds...")
        await asyncio.sleep(5)
        
# Step 2: Go to DuckDuckGo
        print("\n🦆 Step 2: Going to DuckDuckGo...")
        await page.goto("https://duckduckgo.com")
        print("⏳ Waiting 5 seconds (viewing DuckDuckGo homepage)...")
        await asyncio.sleep(5)
        
# Step 3: Type in search box
        print("\n⌨️  Step 3: Typing search query: 'Muhammad Arif Butt'...")
        search_box = page.locator('[name="q"]')   # DuckDuckGo's search box is now a <textarea name="q"> (it used to be an <input>)
        await search_box.press_sequentially("Muhammad Arif Butt", delay=100)   # Type like a human, one key at a time
        print("⏳ Waiting 5 seconds (viewing typed search)...")
        await asyncio.sleep(5)
        
# Step 4: Press Enter
        print("\n🔎 Step 4: Pressing Enter to search...")
        await page.keyboard.press("Enter")
        print("⏳ Waiting for search results to load...")
        await page.wait_for_load_state("domcontentloaded")
        print("⏳ Waiting 10 seconds (viewing search results)...")
        await asyncio.sleep(10)
        
# Step 5: Close browser
        print("\n👋 Step 5: Closing browser...")
        await browser.close()
        print("\n✅ Demo complete!")

# Run the demo
await duckduckgo_search_demo()

🌐 Step 1: Opening Chrome browser...
⏳ Waiting 5 seconds...

🦆 Step 2: Going to DuckDuckGo...
⏳ Waiting 5 seconds (viewing DuckDuckGo homepage)...

⌨️  Step 3: Typing search query: 'Muhammad Arif Butt'...
⏳ Waiting 5 seconds (viewing typed search)...

🔎 Step 4: Pressing Enter to search...
⏳ Waiting for search results to load...
⏳ Waiting 10 seconds (viewing search results)...

👋 Step 5: Closing browser...

✅ Demo complete!
